# 06C FORCE raw vs legacy representation amplification

> Saved outputs are from an earlier local run. Portable path edits have **not** been re-executed. See the repository README for setup and scientific limitations.


In [1]:
import sys
import subprocess
import importlib.util

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "scipy": "scipy",
    "pyarrow": "pyarrow",
}

for module_name, package_name in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {package_name} ...")
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package_name,
            ]
        )

print("All required packages are available.")

All required packages are available.


In [2]:
from pathlib import Path
import hashlib
import json
import platform
import warnings

import numpy as np
import pandas as pd
import scipy

from IPython.display import display
from scipy.stats import spearmanr, wilcoxon

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 300)

RANDOM_STATE = 142
BOOTSTRAP_REPLICATES = 5_000
MINIMUM_WELLS_FOR_STABLE_CONDITION_RANKING = 10

ACTIVE_FEATURES = [
    "GR",
    "NPHI",
    "RHOB",
    "DTC",
    "RDEP",
    "CALI",
]

STRUCTURAL_EXCLUSION_FEATURES = [
    "PEF",
    "SP",
]

ALL_TREATMENT_FEATURES = (
    ACTIVE_FEATURES
    + STRUCTURAL_EXCLUSION_FEATURES
)

DOSES = [
    0.25,
    0.50,
    0.75,
    1.00,
]

In [3]:
# Portable paths: override them with environment variables if upstream artifacts
# live outside this repository. See README -> Reproduction & paths.
import os

def _find_stratashift_root():
    start = Path(os.environ.get("STRATASHIFT_REPO_ROOT", Path.cwd())).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "notebooks" / "interventions").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise FileNotFoundError(
        "StrataShift root not found. Launch Jupyter inside the repository "
        "or set STRATASHIFT_REPO_ROOT."
    )

REPO_ROOT = _find_stratashift_root()
PROJECT_ROOT = Path(os.environ.get(
    "STRATASHIFT_ARTIFACT_ROOT", str(REPO_ROOT / "artifacts")
)).expanduser().resolve()

RAW8_DIR = (
    PROJECT_ROOT
    / "06A_raw_RF_class_conditional_transport_interventions"
    / "full"
)

LEGACY_DIR = (
    PROJECT_ROOT
    / "06B_legacy_RF_class_conditional_transport_interventions"
    / "full"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "06C_raw_vs_legacy_representation_amplification"
    / "full"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RAW8_SUMMARY_PATH = (
    RAW8_DIR
    / "05_condition_dose_effect_summary.csv"
)

RAW8_PER_WELL_PATH = (
    RAW8_DIR
    / "06_all_per_well_effects.csv"
)

RAW8_GATE_PATH = (
    RAW8_DIR
    / "11_stage_gate.json"
)

RAW8_CONFIG_PATH = (
    RAW8_DIR
    / "12_run_configuration.json"
)

LEGACY_SUMMARY_PATH = (
    LEGACY_DIR
    / "05_condition_dose_effect_summary.csv"
)

LEGACY_PER_WELL_PATH = (
    LEGACY_DIR
    / "06_all_per_well_effects.csv"
)

LEGACY_GATE_PATH = (
    LEGACY_DIR
    / "12_stage_gate.json"
)

LEGACY_CONFIG_PATH = (
    LEGACY_DIR
    / "13_run_configuration.json"
)

required_paths = [
    RAW8_SUMMARY_PATH,
    RAW8_PER_WELL_PATH,
    RAW8_GATE_PATH,
    RAW8_CONFIG_PATH,
    LEGACY_SUMMARY_PATH,
    LEGACY_PER_WELL_PATH,
    LEGACY_GATE_PATH,
    LEGACY_CONFIG_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing required inputs:\n"
        + "\n".join(missing_paths)
    )

print("RAW8_DIR:", RAW8_DIR)
print("LEGACY_DIR:", LEGACY_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

RAW8_DIR: <ARTIFACT_ROOT>/06A_raw_RF_class_conditional_transport_interventions/full
LEGACY_DIR: <ARTIFACT_ROOT>/06B_legacy_RF_class_conditional_transport_interventions/full
OUTPUT_DIR: <ARTIFACT_ROOT>/06C_raw_vs_legacy_representation_amplification/full


In [4]:
def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def stable_seed(*parts):
    text = "||".join(
        str(part)
        for part in parts
    )

    return int(
        hashlib.sha256(
            text.encode("utf-8")
        ).hexdigest()[:8],
        16,
    )


def json_safe(value):
    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            json_safe(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    if (
        isinstance(value, float)
        and not np.isfinite(value)
    ):
        return None

    return value


def bootstrap_mean_ci(
    values,
    replicates,
    seed,
    confidence=0.95,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return np.nan, np.nan, np.nan

    if len(values) == 1:
        single = float(
            values[0]
        )

        return (
            single,
            single,
            single,
        )

    rng = np.random.default_rng(
        seed
    )

    bootstrap_indices = rng.integers(
        0,
        len(values),
        size=(
            replicates,
            len(values),
        ),
    )

    bootstrap_means = values[
        bootstrap_indices
    ].mean(axis=1)

    tail = (
        1.0
        - confidence
    ) / 2.0

    return (
        float(
            values.mean()
        ),
        float(
            np.quantile(
                bootstrap_means,
                tail,
            )
        ),
        float(
            np.quantile(
                bootstrap_means,
                1.0 - tail,
            )
        ),
    )


def safe_wilcoxon(
    values,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) < 5:
        return np.nan

    if np.allclose(
        values,
        0.0,
        rtol=0.0,
        atol=1e-15,
    ):
        return 1.0

    try:
        return float(
            wilcoxon(
                values,
                zero_method="wilcox",
                alternative="two-sided",
            ).pvalue
        )

    except ValueError:
        return np.nan


def bh_adjust(
    pvalues,
):
    pvalues = np.asarray(
        pvalues,
        dtype=np.float64,
    )

    output = np.full(
        len(pvalues),
        np.nan,
        dtype=np.float64,
    )

    finite_mask = np.isfinite(
        pvalues
    )

    finite_values = pvalues[
        finite_mask
    ]

    if len(finite_values) == 0:
        return output

    order = np.argsort(
        finite_values
    )

    ranked = finite_values[
        order
    ]

    m = len(ranked)

    adjusted = (
        ranked
        * m
        / np.arange(
            1,
            m + 1,
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]

    adjusted = np.clip(
        adjusted,
        0.0,
        1.0,
    )

    restored = np.empty_like(
        adjusted
    )

    restored[
        order
    ] = adjusted

    output[
        finite_mask
    ] = restored

    return output


def classify_ci(
    lower,
    upper,
):
    if (
        np.isfinite(lower)
        and lower > 0
    ):
        return "amplified"

    if (
        np.isfinite(upper)
        and upper < 0
    ):
        return "suppressed"

    return "inconclusive"


def normalize_boolean_series(
    series,
):
    if pd.api.types.is_bool_dtype(
        series
    ):
        return series.astype(bool)

    normalized = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
    )

    if normalized.isna().any():
        raise ValueError(
            f"Cannot normalize boolean column: {series.name}"
        )

    return normalized.astype(bool)

In [5]:
raw8_summary = pd.read_csv(
    RAW8_SUMMARY_PATH
)

raw8_per_well = pd.read_csv(
    RAW8_PER_WELL_PATH
)

legacy_summary = pd.read_csv(
    LEGACY_SUMMARY_PATH
)

legacy_per_well = pd.read_csv(
    LEGACY_PER_WELL_PATH
)

with open(
    RAW8_GATE_PATH,
    "r",
    encoding="utf-8",
) as file:
    raw8_gate = json.load(
        file
    )

with open(
    RAW8_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as file:
    raw8_config = json.load(
        file
    )

with open(
    LEGACY_GATE_PATH,
    "r",
    encoding="utf-8",
) as file:
    legacy_gate = json.load(
        file
    )

with open(
    LEGACY_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as file:
    legacy_config = json.load(
        file
    )

upstream_gate_audit = {
    "06A_full_passed": bool(
        raw8_gate.get(
            "overall_raw_RF_transport_intervention_stage_passed",
            False,
        )
    ),
    "06B_full_passed": bool(
        legacy_gate.get(
            "overall_legacy_RF_transport_intervention_stage_passed",
            False,
        )
    ),
    "06A_run_mode_full": (
        raw8_gate.get(
            "run_mode"
        )
        == "full"
    ),
    "06B_run_mode_full": (
        legacy_gate.get(
            "run_mode"
        )
        == "full"
    ),
    "06A_condition_count_319": (
        raw8_gate.get(
            "observed_condition_count"
        )
        == 319
    ),
    "06B_condition_count_319": (
        legacy_gate.get(
            "observed_condition_count"
        )
        == 319
    ),
    "06A_condition_dose_count_1276": (
        raw8_gate.get(
            "observed_condition_dose_count"
        )
        == 1276
    ),
    "06B_condition_dose_count_1276": (
        legacy_gate.get(
            "observed_condition_dose_count"
        )
        == 1276
    ),
    "06A_official_test_unused": bool(
        raw8_gate.get(
            "official_test_unused",
            False,
        )
    ),
    "06B_official_test_unused": bool(
        legacy_gate.get(
            "official_test_unused",
            False,
        )
    ),
    "06B_descendants_recomputed": bool(
        legacy_gate.get(
            "descendants_recomputed",
            False,
        )
    ),
    "06B_non_descendants_unchanged": bool(
        legacy_gate.get(
            "non_descendants_unchanged",
            False,
        )
    ),
    "06B_structural_controls_passed": bool(
        legacy_gate.get(
            "PEF_SP_structural_zero_controls_passed",
            False,
        )
    ),
    "06B_legacy_feature_count_26": (
        legacy_config.get(
            "feature_column_count"
        )
        == 26
    ),
    "06B_unique_feature_count_25": (
        legacy_config.get(
            "unique_feature_count"
        )
        == 25
    ),
    "06B_duplicate_X_LOC_preserved": (
        legacy_config.get(
            "duplicate_feature_names"
        )
        == [
            "X_LOC"
        ]
    ),
    "no_new_refit_in_06C": True,
    "official_test_unused_in_06C": True,
}

print(
    json.dumps(
        upstream_gate_audit,
        indent=2,
    )
)

if not all(
    upstream_gate_audit.values()
):
    raise RuntimeError(
        "Upstream gate audit failed."
    )

print(
    "\nRaw-8 summary:",
    raw8_summary.shape,
)

print(
    "Legacy summary:",
    legacy_summary.shape,
)

print(
    "Raw-8 per-well:",
    raw8_per_well.shape,
)

print(
    "Legacy per-well:",
    legacy_per_well.shape,
)

{
  "06A_full_passed": true,
  "06B_full_passed": true,
  "06A_run_mode_full": true,
  "06B_run_mode_full": true,
  "06A_condition_count_319": true,
  "06B_condition_count_319": true,
  "06A_condition_dose_count_1276": true,
  "06B_condition_dose_count_1276": true,
  "06A_official_test_unused": true,
  "06B_official_test_unused": true,
  "06B_descendants_recomputed": true,
  "06B_non_descendants_unchanged": true,
  "06B_structural_controls_passed": true,
  "06B_legacy_feature_count_26": true,
  "06B_unique_feature_count_25": true,
  "06B_duplicate_X_LOC_preserved": true,
  "no_new_refit_in_06C": true,
  "official_test_unused_in_06C": true
}

Raw-8 summary: (1276, 42)
Legacy summary: (1276, 42)
Raw-8 per-well: (20928, 24)
Legacy per-well: (20928, 24)


In [6]:
SUMMARY_KEYS = [
    "condition_id",
    "outer_fold",
    "feature",
    "source_code",
    "target_code",
    "alpha",
]

PER_WELL_KEYS = [
    "WELL",
    "outer_fold",
    "feature",
    "source_code",
    "target_code",
    "alpha",
]

for frame_name, frame, keys in [
    (
        "raw8_summary",
        raw8_summary,
        SUMMARY_KEYS,
    ),
    (
        "legacy_summary",
        legacy_summary,
        SUMMARY_KEYS,
    ),
    (
        "raw8_per_well",
        raw8_per_well,
        PER_WELL_KEYS,
    ),
    (
        "legacy_per_well",
        legacy_per_well,
        PER_WELL_KEYS,
    ),
]:
    duplicated = int(
        frame.duplicated(
            keys
        ).sum()
    )

    if duplicated != 0:
        raise RuntimeError(
            f"Duplicate keys in {frame_name}: {duplicated}"
        )

paired_summary = raw8_summary.merge(
    legacy_summary,
    on=SUMMARY_KEYS,
    how="outer",
    validate="one_to_one",
    indicator=True,
    suffixes=(
        "_raw8",
        "_legacy",
    ),
)

paired_per_well = raw8_per_well.merge(
    legacy_per_well,
    on=PER_WELL_KEYS,
    how="outer",
    validate="one_to_one",
    indicator=True,
    suffixes=(
        "_raw8",
        "_legacy",
    ),
)

summary_pairing_complete = bool(
    (
        paired_summary[
            "_merge"
        ]
        == "both"
    ).all()
)

per_well_pairing_complete = bool(
    (
        paired_per_well[
            "_merge"
        ]
        == "both"
    ).all()
)

if not summary_pairing_complete:
    raise RuntimeError(
        "Condition-dose pairing is incomplete."
    )

if not per_well_pairing_complete:
    raise RuntimeError(
        "Per-well pairing is incomplete."
    )

pairing_audit = {
    "summary_rows": int(
        len(
            paired_summary
        )
    ),
    "per_well_rows": int(
        len(
            paired_per_well
        )
    ),
    "summary_pairing_complete": (
        summary_pairing_complete
    ),
    "per_well_pairing_complete": (
        per_well_pairing_complete
    ),
    "summary_row_counts_match": bool(
        (
            paired_summary[
                "rows_raw8"
            ]
            == paired_summary[
                "rows_legacy"
            ]
        ).all()
    ),
    "summary_well_counts_match": bool(
        (
            paired_summary[
                "wells_raw8"
            ]
            == paired_summary[
                "wells_legacy"
            ]
        ).all()
    ),
    "per_well_eligible_rows_match": bool(
        (
            paired_per_well[
                "rows_raw8"
            ]
            == paired_per_well[
                "rows_legacy"
            ]
        ).all()
    ),
    "summary_source_names_match": bool(
        (
            paired_summary[
                "source_name_raw8"
            ]
            == paired_summary[
                "source_name_legacy"
            ]
        ).all()
    ),
    "summary_target_names_match": bool(
        (
            paired_summary[
                "target_name_raw8"
            ]
            == paired_summary[
                "target_name_legacy"
            ]
        ).all()
    ),
    "per_well_source_names_match": bool(
        (
            paired_per_well[
                "source_name_raw8"
            ]
            == paired_per_well[
                "source_name_legacy"
            ]
        ).all()
    ),
    "per_well_target_names_match": bool(
        (
            paired_per_well[
                "target_name_raw8"
            ]
            == paired_per_well[
                "target_name_legacy"
            ]
        ).all()
    ),
}

print(
    json.dumps(
        pairing_audit,
        indent=2,
    )
)

if not all(
    value
    for key, value in pairing_audit.items()
    if isinstance(
        value,
        bool,
    )
):
    raise RuntimeError(
        "Pairing audit failed."
    )

{
  "summary_rows": 1276,
  "per_well_rows": 20928,
  "summary_pairing_complete": true,
  "per_well_pairing_complete": true,
  "summary_row_counts_match": true,
  "summary_well_counts_match": true,
  "per_well_eligible_rows_match": true,
  "summary_source_names_match": true,
  "summary_target_names_match": true,
  "per_well_source_names_match": true,
  "per_well_target_names_match": true
}


In [7]:
paired_summary = (
    paired_summary.drop(
        columns=[
            "_merge"
        ]
    )
    .copy()
)

paired_per_well = (
    paired_per_well.drop(
        columns=[
            "_merge"
        ]
    )
    .copy()
)

paired_summary[
    "source_name"
] = paired_summary[
    "source_name_raw8"
]

paired_summary[
    "target_name"
] = paired_summary[
    "target_name_raw8"
]

paired_per_well[
    "source_name"
] = paired_per_well[
    "source_name_raw8"
]

paired_per_well[
    "target_name"
] = paired_per_well[
    "target_name_raw8"
]

# ------------------------------------------------------------
# Primary delta-log-loss amplification
# ------------------------------------------------------------
paired_summary[
    "raw_representation_amplification"
] = (
    paired_summary[
        "well_balanced_mean_raw_delta_log_loss_legacy"
    ]
    - paired_summary[
        "well_balanced_mean_raw_delta_log_loss_raw8"
    ]
)

paired_summary[
    "calibrated_representation_amplification"
] = (
    paired_summary[
        "well_balanced_mean_calibrated_delta_log_loss_legacy"
    ]
    - paired_summary[
        "well_balanced_mean_calibrated_delta_log_loss_raw8"
    ]
)

paired_per_well[
    "raw_representation_amplification"
] = (
    paired_per_well[
        "mean_raw_delta_log_loss_legacy"
    ]
    - paired_per_well[
        "mean_raw_delta_log_loss_raw8"
    ]
)

paired_per_well[
    "calibrated_representation_amplification"
] = (
    paired_per_well[
        "mean_calibrated_delta_log_loss_legacy"
    ]
    - paired_per_well[
        "mean_calibrated_delta_log_loss_raw8"
    ]
)

# ------------------------------------------------------------
# Secondary metric differences
# ------------------------------------------------------------
paired_summary[
    "accuracy_change_difference_legacy_minus_raw8"
] = (
    paired_summary[
        "well_balanced_accuracy_change_legacy"
    ]
    - paired_summary[
        "well_balanced_accuracy_change_raw8"
    ]
)

paired_summary[
    "accuracy_harm_amplification"
] = (
    paired_summary[
        "well_balanced_accuracy_change_raw8"
    ]
    - paired_summary[
        "well_balanced_accuracy_change_legacy"
    ]
)

paired_summary[
    "any_flip_amplification"
] = (
    paired_summary[
        "well_balanced_any_flip_rate_legacy"
    ]
    - paired_summary[
        "well_balanced_any_flip_rate_raw8"
    ]
)

paired_summary[
    "targeted_flip_amplification"
] = (
    paired_summary[
        "well_balanced_targeted_flip_rate_legacy"
    ]
    - paired_summary[
        "well_balanced_targeted_flip_rate_raw8"
    ]
)

paired_summary[
    "source_probability_harm_amplification"
] = (
    paired_summary[
        "mean_calibrated_delta_source_probability_raw8"
    ]
    - paired_summary[
        "mean_calibrated_delta_source_probability_legacy"
    ]
)

paired_summary[
    "target_probability_change_difference"
] = (
    paired_summary[
        "mean_calibrated_delta_target_probability_legacy"
    ]
    - paired_summary[
        "mean_calibrated_delta_target_probability_raw8"
    ]
)

paired_per_well[
    "accuracy_change_difference_legacy_minus_raw8"
] = (
    paired_per_well[
        "accuracy_change_legacy"
    ]
    - paired_per_well[
        "accuracy_change_raw8"
    ]
)

paired_per_well[
    "accuracy_harm_amplification"
] = (
    paired_per_well[
        "accuracy_change_raw8"
    ]
    - paired_per_well[
        "accuracy_change_legacy"
    ]
)

paired_per_well[
    "any_flip_amplification"
] = (
    paired_per_well[
        "any_flip_rate_legacy"
    ]
    - paired_per_well[
        "any_flip_rate_raw8"
    ]
)

paired_per_well[
    "targeted_flip_amplification"
] = (
    paired_per_well[
        "targeted_flip_rate_legacy"
    ]
    - paired_per_well[
        "targeted_flip_rate_raw8"
    ]
)

paired_per_well[
    "source_probability_harm_amplification"
] = (
    paired_per_well[
        "mean_calibrated_delta_source_probability_raw8"
    ]
    - paired_per_well[
        "mean_calibrated_delta_source_probability_legacy"
    ]
)

paired_per_well[
    "target_probability_change_difference"
] = (
    paired_per_well[
        "mean_calibrated_delta_target_probability_legacy"
    ]
    - paired_per_well[
        "mean_calibrated_delta_target_probability_raw8"
    ]
)

paired_summary[
    "comparison_scope"
] = np.where(
    paired_summary[
        "feature"
    ].isin(
        STRUCTURAL_EXCLUSION_FEATURES
    ),
    "structural_feature_exclusion",
    "active_representation_comparison",
)

paired_per_well[
    "comparison_scope"
] = np.where(
    paired_per_well[
        "feature"
    ].isin(
        STRUCTURAL_EXCLUSION_FEATURES
    ),
    "structural_feature_exclusion",
    "active_representation_comparison",
)

finite_columns = [
    "raw_representation_amplification",
    "calibrated_representation_amplification",
    "accuracy_harm_amplification",
    "any_flip_amplification",
    "targeted_flip_amplification",
    "source_probability_harm_amplification",
    "target_probability_change_difference",
]

if not np.isfinite(
    paired_per_well[
        finite_columns
    ].to_numpy(float)
).all():
    raise RuntimeError(
        "Non-finite paired per-well effects."
    )

print(
    "Paired condition-dose rows:",
    len(
        paired_summary
    )
)

print(
    "Paired per-well rows:",
    len(
        paired_per_well
    )
)

Paired condition-dose rows: 1276
Paired per-well rows: 20928


# 6. Condition–Dose Representation Amplification

In [8]:
condition_records = []

condition_group_columns = [
    "outer_fold",
    "feature",
    "source_code",
    "source_name",
    "target_code",
    "target_name",
    "alpha",
]

for group_values, frame in paired_per_well.groupby(
    condition_group_columns,
    sort=False,
):
    group_metadata = dict(
        zip(
            condition_group_columns,
            group_values,
        )
    )

    calibrated_values = frame[
        "calibrated_representation_amplification"
    ].to_numpy(float)

    raw_values = frame[
        "raw_representation_amplification"
    ].to_numpy(float)

    (
        calibrated_mean,
        calibrated_lower,
        calibrated_upper,
    ) = bootstrap_mean_ci(
        calibrated_values,
        BOOTSTRAP_REPLICATES,
        stable_seed(
            *group_values,
            "calibrated",
        ),
    )

    (
        raw_mean,
        raw_lower,
        raw_upper,
    ) = bootstrap_mean_ci(
        raw_values,
        BOOTSTRAP_REPLICATES,
        stable_seed(
            *group_values,
            "raw",
        ),
    )

    matching_condition = paired_summary.loc[
        (
            paired_summary[
                "outer_fold"
            ]
            == group_metadata[
                "outer_fold"
            ]
        )
        & (
            paired_summary[
                "feature"
            ]
            == group_metadata[
                "feature"
            ]
        )
        & (
            paired_summary[
                "source_code"
            ]
            == group_metadata[
                "source_code"
            ]
        )
        & (
            paired_summary[
                "target_code"
            ]
            == group_metadata[
                "target_code"
            ]
        )
        & np.isclose(
            paired_summary[
                "alpha"
            ],
            group_metadata[
                "alpha"
            ],
        )
    ]

    if len(
        matching_condition
    ) != 1:
        raise RuntimeError(
            "Condition lookup failed."
        )

    condition_row = (
        matching_condition.iloc[0]
    )

    condition_records.append({
        "condition_id": condition_row[
            "condition_id"
        ],
        **group_metadata,
        "rows": int(
            frame[
                "rows_raw8"
            ].sum()
        ),
        "wells": int(
            frame[
                "WELL"
            ].nunique()
        ),
        "raw8_well_balanced_calibrated_delta_log_loss": float(
            frame[
                "mean_calibrated_delta_log_loss_raw8"
            ].mean()
        ),
        "legacy_well_balanced_calibrated_delta_log_loss": float(
            frame[
                "mean_calibrated_delta_log_loss_legacy"
            ].mean()
        ),
        "well_balanced_calibrated_representation_amplification": float(
            calibrated_mean
        ),
        "calibrated_CI_lower": float(
            calibrated_lower
        ),
        "calibrated_CI_upper": float(
            calibrated_upper
        ),
        "calibrated_wilcoxon_pvalue": safe_wilcoxon(
            calibrated_values
        ),
        "median_per_well_calibrated_amplification": float(
            np.median(
                calibrated_values
            )
        ),
        "fraction_wells_positive_calibrated_amplification": float(
            (
                calibrated_values
                > 0
            ).mean()
        ),
        "well_balanced_raw_representation_amplification": float(
            raw_mean
        ),
        "raw_CI_lower": float(
            raw_lower
        ),
        "raw_CI_upper": float(
            raw_upper
        ),
        "raw_wilcoxon_pvalue": safe_wilcoxon(
            raw_values
        ),
        "well_balanced_accuracy_harm_amplification": float(
            frame[
                "accuracy_harm_amplification"
            ].mean()
        ),
        "well_balanced_any_flip_amplification": float(
            frame[
                "any_flip_amplification"
            ].mean()
        ),
        "well_balanced_targeted_flip_amplification": float(
            frame[
                "targeted_flip_amplification"
            ].mean()
        ),
        "well_balanced_source_probability_harm_amplification": float(
            frame[
                "source_probability_harm_amplification"
            ].mean()
        ),
        "well_balanced_target_probability_change_difference": float(
            frame[
                "target_probability_change_difference"
            ].mean()
        ),
        "comparison_scope": (
            "structural_feature_exclusion"
            if group_metadata[
                "feature"
            ]
            in STRUCTURAL_EXCLUSION_FEATURES
            else "active_representation_comparison"
        ),
        "stable_condition_support": bool(
            frame[
                "WELL"
            ].nunique()
            >= MINIMUM_WELLS_FOR_STABLE_CONDITION_RANKING
        ),
        "bootstrap_replicates": int(
            BOOTSTRAP_REPLICATES
        ),
    })

condition_amplification = pd.DataFrame(
    condition_records
)

for alpha in DOSES:
    alpha_mask = np.isclose(
        condition_amplification[
            "alpha"
        ],
        alpha,
    )

    condition_amplification.loc[
        alpha_mask,
        "calibrated_qvalue",
    ] = bh_adjust(
        condition_amplification.loc[
            alpha_mask,
            "calibrated_wilcoxon_pvalue",
        ].to_numpy(float)
    )

    condition_amplification.loc[
        alpha_mask,
        "raw_qvalue",
    ] = bh_adjust(
        condition_amplification.loc[
            alpha_mask,
            "raw_wilcoxon_pvalue",
        ].to_numpy(float)
    )

condition_amplification[
    "calibrated_FDR_significant"
] = (
    condition_amplification[
        "calibrated_qvalue"
    ]
    < 0.05
)

condition_amplification[
    "calibrated_amplification_class"
] = [
    classify_ci(
        lower,
        upper,
    )
    for lower, upper in zip(
        condition_amplification[
            "calibrated_CI_lower"
        ],
        condition_amplification[
            "calibrated_CI_upper"
        ],
    )
]

condition_amplification.loc[
    condition_amplification[
        "comparison_scope"
    ]
    == "structural_feature_exclusion",
    "calibrated_amplification_class",
] = "structural_exclusion"

display(
    condition_amplification.loc[
        (
            np.isclose(
                condition_amplification[
                    "alpha"
                ],
                1.0,
            )
        )
        & (
            condition_amplification[
                "comparison_scope"
            ]
            == "active_representation_comparison"
        )
        & (
            condition_amplification[
                "stable_condition_support"
            ]
        )
    ]
    .sort_values(
        "well_balanced_calibrated_representation_amplification",
        ascending=False,
    )
    .head(20)
)

,condition_id,outer_fold,feature,source_code,source_name,target_code,target_name,alpha,rows,wells,raw8_well_balanced_calibrated_delta_log_loss,legacy_well_balanced_calibrated_delta_log_loss,well_balanced_calibrated_representation_amplification,calibrated_CI_lower,calibrated_CI_upper,calibrated_wilcoxon_pvalue,median_per_well_calibrated_amplification,fraction_wells_positive_calibrated_amplification,well_balanced_raw_representation_amplification,raw_CI_lower,raw_CI_upper,raw_wilcoxon_pvalue,well_balanced_accuracy_harm_amplification,well_balanced_any_flip_amplification,well_balanced_targeted_flip_amplification,well_balanced_source_probability_harm_amplification,well_balanced_target_probability_change_difference,comparison_scope,stable_condition_support,bootstrap_replicates,calibrated_qvalue,raw_qvalue,calibrated_FDR_significant,calibrated_amplification_class
63,fold_1__feature_GR__source_30000_Sandstone__target_65000_Shale,1,GR,30000,Sandstone,65000,Shale,1.0,32288,20,0.484601,0.829650,0.345049,0.184804,0.507328,0.000586,0.253857,0.800000,0.149464,0.009156,0.294006,0.105398,0.129883,0.150094,0.125755,0.083494,0.087547,active_representation_comparison,True,5000,0.010884,0.266447,True,amplified
1199,fold_4__feature_RHOB__source_70000_Limestone__target_65000_Shale,4,RHOB,70000,Limestone,65000,Shale,1.0,11285,17,0.149134,0.449883,0.300749,0.198341,0.406310,0.000015,0.267955,1.000000,0.197254,0.106493,0.294111,0.000504,0.084596,0.005977,0.006663,0.051164,0.027025,active_representation_comparison,True,5000,0.001607,0.009252,True,amplified
883,fold_2__feature_NPHI__source_70000_Limestone__target_65000_Shale,2,NPHI,70000,Limestone,65000,Shale,1.0,7635,20,-0.082191,0.212357,0.294548,0.154526,0.464646,0.000395,0.268648,0.750000,0.274353,0.133197,0.446961,0.001432,0.046396,-0.073124,0.015777,0.046603,0.000065,active_representation_comparison,True,5000,0.008912,0.012933,True,amplified
571,fold_5__feature_GR__source_30000_Sandstone__target_65000_Shale,5,GR,30000,Sandstone,65000,Shale,1.0,32871,20,0.856559,1.108029,0.251470,0.133381,0.360989,0.000708,0.337681,0.850000,0.004405,-0.115751,0.116219,0.647655,0.085379,0.074371,0.065785,0.029350,0.054279,active_representation_comparison,True,5000,0.011688,0.824562,True,amplified
967,fold_2__feature_RHOB__source_70000_Limestone__target_65000_Shale,2,RHOB,70000,Limestone,65000,Shale,1.0,9447,20,0.220337,0.467757,0.247420,0.117663,0.387676,0.000851,0.145501,0.850000,0.187710,0.073364,0.306814,0.008308,0.077440,0.057158,0.002703,0.056209,0.008264,active_representation_comparison,True,5000,0.011688,0.041023,True,amplified
887,fold_2__feature_NPHI__source_70000_Limestone__target_74000_Dolomite,2,NPHI,70000,Limestone,74000,Dolomite,1.0,7635,20,-0.092429,0.144714,0.237143,0.105898,0.409025,0.000851,0.151456,0.800000,0.219557,0.087462,0.390827,0.003153,0.034113,-0.070515,0.000000,0.033116,-0.002060,active_representation_comparison,True,5000,0.011688,0.021659,True,amplified
1051,fold_4__feature_DTC__source_30000_Sandstone__target_65000_Shale,4,DTC,30000,Sandstone,65000,Shale,1.0,31677,17,0.138307,0.333549,0.195243,0.097406,0.291693,0.000839,0.102529,0.941176,0.171985,0.083523,0.257849,0.001068,0.024262,0.011518,0.028231,0.059068,0.041387,active_representation_comparison,True,5000,0.011688,0.011251,True,amplified
1095,fold_4__feature_GR__source_65000_Shale__target_70000_Limestone,4,GR,65000,Shale,70000,Limestone,1.0,142571,17,0.872009,1.051579,0.179570,0.063477,0.306624,0.010986,0.202877,0.705882,0.035345,-0.073292,0.147018,0.611221,0.106116,0.070577,-0.002101,0.070380,0.015217,active_representation_comparison,True,5000,0.057861,0.808142,False,amplified
935,fold_2__feature_RDEP__source_70000_Limestone__target_65000_Shale,2,RDEP,70000,Limestone,65000,Shale,1.0,9844,20,-0.165184,0.013743,0.178927,0.082639,0.292995,0.001432,0.156225,0.800000,0.180623,0.086398,0.288177,0.001209,0.008391,-0.057358,-0.002213,0.020060,0.010760,active_representation_comparison,True,5000,0.014601,0.011941,True,amplified
623,fold_5__fea

# 7. Feature–Dose Well-Balanced Amplification

In [9]:
# First average all available directed class-pair effects
# within each WELL for each Feature and Dose.
per_well_feature_alpha = (
    paired_per_well.groupby(
        [
            "WELL",
            "feature",
            "alpha",
        ],
        as_index=False,
    )
    .agg(
        directed_condition_count=(
            "source_code",
            "size",
        ),
        mean_calibrated_representation_amplification=(
            "calibrated_representation_amplification",
            "mean",
        ),
        median_calibrated_representation_amplification=(
            "calibrated_representation_amplification",
            "median",
        ),
        mean_raw_representation_amplification=(
            "raw_representation_amplification",
            "mean",
        ),
        mean_accuracy_harm_amplification=(
            "accuracy_harm_amplification",
            "mean",
        ),
        mean_any_flip_amplification=(
            "any_flip_amplification",
            "mean",
        ),
        mean_targeted_flip_amplification=(
            "targeted_flip_amplification",
            "mean",
        ),
        mean_source_probability_harm_amplification=(
            "source_probability_harm_amplification",
            "mean",
        ),
        mean_target_probability_change_difference=(
            "target_probability_change_difference",
            "mean",
        ),
    )
)

feature_records = []

for (
    feature,
    alpha,
), frame in per_well_feature_alpha.groupby(
    [
        "feature",
        "alpha",
    ],
    sort=False,
):
    calibrated_values = frame[
        "mean_calibrated_representation_amplification"
    ].to_numpy(float)

    raw_values = frame[
        "mean_raw_representation_amplification"
    ].to_numpy(float)

    (
        calibrated_mean,
        calibrated_lower,
        calibrated_upper,
    ) = bootstrap_mean_ci(
        calibrated_values,
        BOOTSTRAP_REPLICATES,
        stable_seed(
            feature,
            alpha,
            "feature_calibrated",
        ),
    )

    (
        raw_mean,
        raw_lower,
        raw_upper,
    ) = bootstrap_mean_ci(
        raw_values,
        BOOTSTRAP_REPLICATES,
        stable_seed(
            feature,
            alpha,
            "feature_raw",
        ),
    )

    condition_subset = condition_amplification.loc[
        (
            condition_amplification[
                "feature"
            ]
            == feature
        )
        & np.isclose(
            condition_amplification[
                "alpha"
            ],
            alpha,
        )
    ]

    feature_records.append({
        "feature": feature,
        "alpha": float(
            alpha
        ),
        "wells": int(
            frame[
                "WELL"
            ].nunique()
        ),
        "directed_conditions": int(
            condition_subset[
                "condition_id"
            ].nunique()
        ),
        "mean_well_balanced_calibrated_representation_amplification": float(
            calibrated_mean
        ),
        "calibrated_CI_lower": float(
            calibrated_lower
        ),
        "calibrated_CI_upper": float(
            calibrated_upper
        ),
        "calibrated_wilcoxon_pvalue": safe_wilcoxon(
            calibrated_values
        ),
        "median_per_well_calibrated_amplification": float(
            np.median(
                calibrated_values
            )
        ),
        "fraction_wells_positive_calibrated_amplification": float(
            (
                calibrated_values
                > 0
            ).mean()
        ),
        "mean_well_balanced_raw_representation_amplification": float(
            raw_mean
        ),
        "raw_CI_lower": float(
            raw_lower
        ),
        "raw_CI_upper": float(
            raw_upper
        ),
        "raw_wilcoxon_pvalue": safe_wilcoxon(
            raw_values
        ),
        "mean_accuracy_harm_amplification": float(
            frame[
                "mean_accuracy_harm_amplification"
            ].mean()
        ),
        "mean_any_flip_amplification": float(
            frame[
                "mean_any_flip_amplification"
            ].mean()
        ),
        "mean_targeted_flip_amplification": float(
            frame[
                "mean_targeted_flip_amplification"
            ].mean()
        ),
        "mean_source_probability_harm_amplification": float(
            frame[
                "mean_source_probability_harm_amplification"
            ].mean()
        ),
        "mean_target_probability_change_difference": float(
            frame[
                "mean_target_probability_change_difference"
            ].mean()
        ),
        "median_condition_amplification": float(
            condition_subset[
                "well_balanced_calibrated_representation_amplification"
            ].median()
        ),
        "median_absolute_condition_amplification": float(
            condition_subset[
                "well_balanced_calibrated_representation_amplification"
            ].abs().median()
        ),
        "maximum_condition_amplification": float(
            condition_subset[
                "well_balanced_calibrated_representation_amplification"
            ].max()
        ),
        "minimum_condition_amplification": float(
            condition_subset[
                "well_balanced_calibrated_representation_amplification"
            ].min()
        ),
        "comparison_scope": (
            "structural_feature_exclusion"
            if feature
            in STRUCTURAL_EXCLUSION_FEATURES
            else "active_representation_comparison"
        ),
        "bootstrap_replicates": int(
            BOOTSTRAP_REPLICATES
        ),
    })

feature_amplification = pd.DataFrame(
    feature_records
)

for alpha in DOSES:
    alpha_mask = np.isclose(
        feature_amplification[
            "alpha"
        ],
        alpha,
    )

    feature_amplification.loc[
        alpha_mask,
        "calibrated_qvalue_all_features",
    ] = bh_adjust(
        feature_amplification.loc[
            alpha_mask,
            "calibrated_wilcoxon_pvalue",
        ].to_numpy(float)
    )

    active_mask = (
        alpha_mask
        & (
            feature_amplification[
                "comparison_scope"
            ]
            == "active_representation_comparison"
        )
    )

    feature_amplification.loc[
        active_mask,
        "calibrated_qvalue_active_features",
    ] = bh_adjust(
        feature_amplification.loc[
            active_mask,
            "calibrated_wilcoxon_pvalue",
        ].to_numpy(float)
    )

feature_amplification[
    "calibrated_amplification_class"
] = [
    classify_ci(
        lower,
        upper,
    )
    for lower, upper in zip(
        feature_amplification[
            "calibrated_CI_lower"
        ],
        feature_amplification[
            "calibrated_CI_upper"
        ],
    )
]

feature_amplification.loc[
    feature_amplification[
        "comparison_scope"
    ]
    == "structural_feature_exclusion",
    "calibrated_amplification_class",
] = "structural_exclusion"

display(
    feature_amplification.loc[
        np.isclose(
            feature_amplification[
                "alpha"
            ],
            1.0,
        )
    ]
    .sort_values(
        "mean_well_balanced_calibrated_representation_amplification",
        ascending=False,
    )
)

,feature,alpha,wells,directed_conditions,mean_well_balanced_calibrated_representation_amplification,calibrated_CI_lower,calibrated_CI_upper,calibrated_wilcoxon_pvalue,median_per_well_calibrated_amplification,fraction_wells_positive_calibrated_amplification,mean_well_balanced_raw_representation_amplification,raw_CI_lower,raw_CI_upper,raw_wilcoxon_pvalue,mean_accuracy_harm_amplification,mean_any_flip_amplification,mean_targeted_flip_amplification,mean_source_probability_harm_amplification,mean_target_probability_change_difference,median_condition_amplification,median_absolute_condition_amplification,maximum_condition_amplification,minimum_condition_amplification,comparison_scope,bootstrap_replicates,calibrated_qvalue_all_features,calibrated_qvalue_active_features,calibrated_amplification_class
7,DTC,1.0,98,40,0.079674,0.017252,0.142138,0.000025,0.067552,0.693878,0.099248,0.034596,0.163729,3.316306e-07,0.033334,0.015142,0.009566,0.028213,0.010820,0.055527,0.058708,2.636640,-0.886764,active_representation_comparison,5000,0.000068,0.000076,amplified
27,RHOB,1.0,98,40,0.056760,0.018492,0.105177,0.000757,0.036575,0.683673,0.053733,0.015749,0.101359,2.999924e-03,0.016071,-0.008786,-0.000401,0.005964,0.004225,0.011080,0.031947,2.173786,-0.386674,active_representation_comparison,5000,0.001009,0.000908,amplified
11,GR,1.0,98,40,0.018175,-0.038072,0.067581,0.056363,0.037993,0.540816,-0.063358,-0.121125,-0.013263,3.334388e-02,0.020884,0.007255,0.018458,0.028858,0.021091,0.017523,0.066598,0.768755,-1.118033,active_representation_comparison,5000,0.056363,0.056363,inconclusive
31,SP,1.0,68,40,-0.016160,-0.040104,0.007410,0.042495,-0.009107,0.382353,-0.016687,-0.041041,0.009371,4.066119e-02,-0.001220,-0.039251,-0.003413,-0.000752,-0.001621,-0.002830,0.014945,0.185298,-0.804766,structural_feature_exclusion,5000,0.048566,NaN,structural_exclusion
3,CALI,1.0,97,40,-0.039300,-0.083815,0.018638,0.000001,-0.048239,0.247423,-0.038967,-0.086827,0.020691,8.628588e-07,-0.011308,-0.070756,-0.010204,-0.015207,-0.006006,-0.019990,0.026358,1.824781,-1.204268,active_representation_comparison,5000,0.000005,0.000007,inconclusive
19,PEF,1.0,69,39,-0.045101,-0.072049,-0.020892,0.000001,-0.039251,0.130435,-0.045442,-0.074449,-0.020434,1.601743e-06,-0.012700,-0.049843,-0.005577,-0.012843,-0.002843,-0.018521,0.030464,0.971259,-0.462959,structural_feature_exclusion,5000,0.000005,NaN,structural_exclusion
15,NPHI,1.0,98,40,-0.061079,-0.096171,-0.028009,0.000436,-0.045312,0.346939,-0.074695,-0.110417,-0.042100,1.177931e-05,-0.032708,-0.073167,-0.024717,-0.025875,-0.016530,-0.089065,0.110182,0.294548,-1.660990,active_representation_comparison,5000,0.000698,0.000655,suppressed
23,RDEP,1.0,98,40,-0.085766,-0.165783,-0.030753,0.000233,-0.033295,0.346939,-0.086964,-0.166437,-0.030472,3.072219e-04,-0.006375,-0.053116,-0.006694,-0.007209,-0.001910,-0.020921,0.035376,0.178927,-3.831665,active_representation_comparison,5000,0.000466,0.000466,suppressed


In [10]:
fold_feature_alpha = (
    paired_per_well.groupby(
        [
            "outer_fold",
            "WELL",
            "feature",
            "alpha",
        ],
        as_index=False,
    )
    .agg(
        mean_calibrated_representation_amplification=(
            "calibrated_representation_amplification",
            "mean",
        ),
        mean_raw_representation_amplification=(
            "raw_representation_amplification",
            "mean",
        ),
        mean_accuracy_harm_amplification=(
            "accuracy_harm_amplification",
            "mean",
        ),
        mean_any_flip_amplification=(
            "any_flip_amplification",
            "mean",
        ),
        mean_targeted_flip_amplification=(
            "targeted_flip_amplification",
            "mean",
        ),
    )
    .groupby(
        [
            "outer_fold",
            "feature",
            "alpha",
        ],
        as_index=False,
    )
    .agg(
        wells=(
            "WELL",
            "nunique",
        ),
        mean_calibrated_representation_amplification=(
            "mean_calibrated_representation_amplification",
            "mean",
        ),
        median_calibrated_representation_amplification=(
            "mean_calibrated_representation_amplification",
            "median",
        ),
        mean_raw_representation_amplification=(
            "mean_raw_representation_amplification",
            "mean",
        ),
        mean_accuracy_harm_amplification=(
            "mean_accuracy_harm_amplification",
            "mean",
        ),
        mean_any_flip_amplification=(
            "mean_any_flip_amplification",
            "mean",
        ),
        mean_targeted_flip_amplification=(
            "mean_targeted_flip_amplification",
            "mean",
        ),
    )
)

dose_response_records = []

for feature, frame in feature_amplification.groupby(
    "feature"
):
    frame = frame.sort_values(
        "alpha"
    )

    calibrated_test = spearmanr(
        frame[
            "alpha"
        ].to_numpy(float),
        frame[
            "mean_well_balanced_calibrated_representation_amplification"
        ].to_numpy(float),
    )

    raw_test = spearmanr(
        frame[
            "alpha"
        ].to_numpy(float),
        frame[
            "mean_well_balanced_raw_representation_amplification"
        ].to_numpy(float),
    )

    dose_response_records.append({
        "feature": feature,
        "comparison_scope": frame[
            "comparison_scope"
        ].iloc[0],
        "calibrated_spearman_alpha_vs_amplification": float(
            calibrated_test.statistic
        ),
        "calibrated_spearman_pvalue": float(
            calibrated_test.pvalue
        ),
        "raw_spearman_alpha_vs_amplification": float(
            raw_test.statistic
        ),
        "raw_spearman_pvalue": float(
            raw_test.pvalue
        ),
        "calibrated_alpha1_minus_alpha025": float(
            frame.loc[
                np.isclose(
                    frame[
                        "alpha"
                    ],
                    1.0,
                ),
                "mean_well_balanced_calibrated_representation_amplification",
            ].iloc[0]
            - frame.loc[
                np.isclose(
                    frame[
                        "alpha"
                    ],
                    0.25,
                ),
                "mean_well_balanced_calibrated_representation_amplification",
            ].iloc[0]
        ),
    })

dose_response_summary = pd.DataFrame(
    dose_response_records
)

display(
    fold_feature_alpha.loc[
        np.isclose(
            fold_feature_alpha[
                "alpha"
            ],
            1.0,
        )
    ]
    .sort_values(
        [
            "feature",
            "outer_fold",
        ]
    )
)

,outer_fold,feature,alpha,wells,mean_calibrated_representation_amplification,median_calibrated_representation_amplification,mean_raw_representation_amplification,mean_accuracy_harm_amplification,mean_any_flip_amplification,mean_targeted_flip_amplification
3,1,CALI,1.0,20,0.034725,-0.045491,0.042460,-0.010793,-0.053859,-0.006990
35,2,CALI,1.0,20,-0.096708,-0.041316,-0.097883,-0.004395,-0.052798,-0.002710
67,3,CALI,1.0,20,-0.087709,-0.056963,-0.090587,-0.017667,-0.078343,-0.013847
99,4,CALI,1.0,17,-0.027051,-0.053107,-0.029441,-0.015319,-0.102073,-0.013668
131,5,CALI,1.0,20,-0.017922,-0.037899,-0.017956,-0.008966,-0.071404,-0.014324
7,1,DTC,1.0,20,0.059714,0.026528,0.084083,0.036070,0.029667,0.006993
39,2,DTC,1.0,20,0.058429,0.060641,0.067436,0.045754,0.035219,0.020036
71,3,DTC,1.0,21,0.003061,0.040143,0.021205,0.038800,0.014086,0.013353
103,4,DTC,1.0,17,0.185773,0.158481,0.206829,0.023204,-0.000140,0.004449
135,5,DTC,1.0,20,0.111136,0.076041,0.136725,0.021047,-0.005363,0.002039


# 9. Structural exclusion controls

In [11]:
legacy_structural_summary = legacy_summary.loc[
    legacy_summary[
        "feature"
    ].isin(
        STRUCTURAL_EXCLUSION_FEATURES
    )
].copy()

structural_metric_columns = [
    "well_balanced_mean_raw_delta_log_loss",
    "well_balanced_mean_calibrated_delta_log_loss",
    "pooled_mean_raw_delta_log_loss",
    "pooled_mean_calibrated_delta_log_loss",
    "pooled_accuracy_change",
    "well_balanced_accuracy_change",
    "pooled_any_flip_rate",
    "well_balanced_any_flip_rate",
    "pooled_targeted_flip_rate",
    "well_balanced_targeted_flip_rate",
]

structural_exact_zero = bool(
    (
        legacy_structural_summary[
            structural_metric_columns
        ]
        .abs()
        .to_numpy(float)
        == 0.0
    ).all()
)

structural_exclusion_summary = (
    feature_amplification.loc[
        feature_amplification[
            "feature"
        ].isin(
            STRUCTURAL_EXCLUSION_FEATURES
        )
    ]
    .copy()
)

structural_exclusion_summary[
    "legacy_effect_is_exact_zero"
] = structural_exact_zero

structural_exclusion_summary[
    "interpretation"
] = (
    "Feature is present in Raw-8 but absent from the final "
    "Legacy representation; difference is structural exclusion, "
    "not engineered descendant amplification."
)

print(
    "Legacy PEF/SP exact-zero effects:",
    structural_exact_zero,
)

display(
    structural_exclusion_summary
)

Legacy PEF/SP exact-zero effects: True


,feature,alpha,wells,directed_conditions,mean_well_balanced_calibrated_representation_amplification,calibrated_CI_lower,calibrated_CI_upper,calibrated_wilcoxon_pvalue,median_per_well_calibrated_amplification,fraction_wells_positive_calibrated_amplification,mean_well_balanced_raw_representation_amplification,raw_CI_lower,raw_CI_upper,raw_wilcoxon_pvalue,mean_accuracy_harm_amplification,mean_any_flip_amplification,mean_targeted_flip_amplification,mean_source_probability_harm_amplification,mean_target_probability_change_difference,median_condition_amplification,median_absolute_condition_amplification,maximum_condition_amplification,minimum_condition_amplification,comparison_scope,bootstrap_replicates,calibrated_qvalue_all_features,calibrated_qvalue_active_features,calibrated_amplification_class,legacy_effect_is_exact_zero,interpretation
16,PEF,0.25,69,39,-0.009479,-0.018525,-0.000128,2.511506e-07,-0.007582,0.144928,-0.009264,-0.019104,0.000270,2.944792e-07,-0.003614,-0.016990,-0.001542,-0.002606,-0.000577,-0.005739,0.007551,0.639260,-0.339549,structural_feature_exclusion,5000,2.009205e-06,NaN,structural_exclusion,True,"Feature is present in Raw-8 but absent from the final Legacy representation; difference is structural exclusion, not engineered descendant amplification."
17,PEF,0.50,69,39,-0.016198,-0.034783,0.001084,4.331224e-06,-0.014606,0.173913,-0.015861,-0.035151,0.002427,7.016803e-06,-0.006111,-0.028953,-0.002898,-0.005730,-0.001239,-0.008547,0.015646,0.987494,-0.402752,structural_feature_exclusion,5000,3.464980e-05,NaN,structural_exclusion,True,"Feature is present in Raw-8 but absent from the final Legacy representation; difference is structural exclusion, not engineered descendant amplification."
18,PEF,0.75,69,39,-0.033872,-0.058210,-0.011990,9.808811e-08,-0.026477,0.115942,-0.033962,-0.059740,-0.012480,1.194359e-07,-0.008881,-0.039354,-0.004093,-0.009194,-0.002050,-0.013794,0.022746,1.023248,-0.505596,structural_feature_exclusion,5000,7.847049e-07,NaN,structural_exclusion,True,"Feature is present in Raw-8 but absent from the final Legacy representation; difference is structural exclusion, not engineered descendant amplification."
19,PEF,1.00,69,39,-0.045101,-0.072049,-0.020892,1.298672e-06,-0.039251,0.130435,-0.045442,-0.074449,-0.020434,1.601743e-06,-0.012700,-0.049843,-0.005577,-0.012843,-0.002843,-0.018521,0.030464,0.971259,-0.462959,structural_feature_exclusion,5000,5.194687e-06,NaN,structural_exclusion,True,"Feature is present in Raw-8 but absent from the final Legacy representation; difference is structural exclusion, not engineered descendant amplification."
28,SP,0.25,68,40,-0.009083,-0.017067,-0.002255,4.322574e-04,-0.002960,0.352941,-0.009156,-0.017402,-0.002131,3.427180e-04,-0.000467,-0.011643,-0.000798,-0.000625,-0.000244,-0.002684,0.004687,0.026755,-0.426096,structural_feature_exclusion,5000,1.152686e-03,NaN,structural_exclusion,True,"Feature is present in Raw-8 but absent from the final Legacy representation; difference is structural exclusion, not engineered descendant amplification."
29,SP,0.50,68,40,-0.013374,-0.024827,-0.002540,2.159315e-03,-0.005481,0.367647,-0.013526,-0.025355,-0.002598,2.030541e-03,-0.001557,-0.021064,-0.001521,-0.001177,-0.000732,-0.004368,0.007953,0.154641,-0.428207,structural_feature_exclusion,5000,3.454905e-03,NaN,structural_exclusion,True,"Feature is present in Raw-8 but absent from the final Legacy representation; difference is structural exclusion, not engineered descendant amplification."
30,SP,0.75,68,40,-0.014452,-0.034284,0.003575,1.223666e-02,-0.010262,0.367647,-0.014755,-0.034748,0.004460,1.202686e-02,-0.001619,-0.029750,-0.002420,-0.001137,-0.001116,-0.005462,0.010185,0.089399,-0.800993,structural_feature_exclusion,5000,1.398476e-02,NaN,structural_exclusion,True,"Feature is present in Raw-8 but absent from the final Legacy representation; difference is structural exclusion, not engineered descendant amplification."
31,SP,1.00,68,40,-0.016160,-0.040104,0.007410,4.249524

In [12]:
alpha_one_active = condition_amplification.loc[
    (
        np.isclose(
            condition_amplification[
                "alpha"
            ],
            1.0,
        )
    )
    & (
        condition_amplification[
            "comparison_scope"
        ]
        == "active_representation_comparison"
    )
].copy()

top_amplified_alpha1_all = (
    alpha_one_active.sort_values(
        "well_balanced_calibrated_representation_amplification",
        ascending=False,
    )
)

top_suppressed_alpha1_all = (
    alpha_one_active.sort_values(
        "well_balanced_calibrated_representation_amplification",
        ascending=True,
    )
)

top_amplified_alpha1_stable = (
    top_amplified_alpha1_all.loc[
        top_amplified_alpha1_all[
            "stable_condition_support"
        ]
    ]
)

top_suppressed_alpha1_stable = (
    top_suppressed_alpha1_all.loc[
        top_suppressed_alpha1_all[
            "stable_condition_support"
        ]
    ]
)

print(
    "Top amplified active conditions with at least",
    MINIMUM_WELLS_FOR_STABLE_CONDITION_RANKING,
    "wells:",
)

display(
    top_amplified_alpha1_stable.head(20)
)

print(
    "Top suppressed active conditions with at least",
    MINIMUM_WELLS_FOR_STABLE_CONDITION_RANKING,
    "wells:",
)

display(
    top_suppressed_alpha1_stable.head(20)
)

Top amplified active conditions with at least 10 wells:


,condition_id,outer_fold,feature,source_code,source_name,target_code,target_name,alpha,rows,wells,raw8_well_balanced_calibrated_delta_log_loss,legacy_well_balanced_calibrated_delta_log_loss,well_balanced_calibrated_representation_amplification,calibrated_CI_lower,calibrated_CI_upper,calibrated_wilcoxon_pvalue,median_per_well_calibrated_amplification,fraction_wells_positive_calibrated_amplification,well_balanced_raw_representation_amplification,raw_CI_lower,raw_CI_upper,raw_wilcoxon_pvalue,well_balanced_accuracy_harm_amplification,well_balanced_any_flip_amplification,well_balanced_targeted_flip_amplification,well_balanced_source_probability_harm_amplification,well_balanced_target_probability_change_difference,comparison_scope,stable_condition_support,bootstrap_replicates,calibrated_qvalue,raw_qvalue,calibrated_FDR_significant,calibrated_amplification_class
63,fold_1__feature_GR__source_30000_Sandstone__target_65000_Shale,1,GR,30000,Sandstone,65000,Shale,1.0,32288,20,0.484601,0.829650,0.345049,0.184804,0.507328,0.000586,0.253857,0.800000,0.149464,0.009156,0.294006,0.105398,0.129883,0.150094,0.125755,0.083494,0.087547,active_representation_comparison,True,5000,0.010884,0.266447,True,amplified
1199,fold_4__feature_RHOB__source_70000_Limestone__target_65000_Shale,4,RHOB,70000,Limestone,65000,Shale,1.0,11285,17,0.149134,0.449883,0.300749,0.198341,0.406310,0.000015,0.267955,1.000000,0.197254,0.106493,0.294111,0.000504,0.084596,0.005977,0.006663,0.051164,0.027025,active_representation_comparison,True,5000,0.001607,0.009252,True,amplified
883,fold_2__feature_NPHI__source_70000_Limestone__target_65000_Shale,2,NPHI,70000,Limestone,65000,Shale,1.0,7635,20,-0.082191,0.212357,0.294548,0.154526,0.464646,0.000395,0.268648,0.750000,0.274353,0.133197,0.446961,0.001432,0.046396,-0.073124,0.015777,0.046603,0.000065,active_representation_comparison,True,5000,0.008912,0.012933,True,amplified
571,fold_5__feature_GR__source_30000_Sandstone__target_65000_Shale,5,GR,30000,Sandstone,65000,Shale,1.0,32871,20,0.856559,1.108029,0.251470,0.133381,0.360989,0.000708,0.337681,0.850000,0.004405,-0.115751,0.116219,0.647655,0.085379,0.074371,0.065785,0.029350,0.054279,active_representation_comparison,True,5000,0.011688,0.824562,True,amplified
967,fold_2__feature_RHOB__source_70000_Limestone__target_65000_Shale,2,RHOB,70000,Limestone,65000,Shale,1.0,9447,20,0.220337,0.467757,0.247420,0.117663,0.387676,0.000851,0.145501,0.850000,0.187710,0.073364,0.306814,0.008308,0.077440,0.057158,0.002703,0.056209,0.008264,active_representation_comparison,True,5000,0.011688,0.041023,True,amplified
887,fold_2__feature_NPHI__source_70000_Limestone__target_74000_Dolomite,2,NPHI,70000,Limestone,74000,Dolomite,1.0,7635,20,-0.092429,0.144714,0.237143,0.105898,0.409025,0.000851,0.151456,0.800000,0.219557,0.087462,0.390827,0.003153,0.034113,-0.070515,0.000000,0.033116,-0.002060,active_representation_comparison,True,5000,0.011688,0.021659,True,amplified
1051,fold_4__feature_DTC__source_30000_Sandstone__target_65000_Shale,4,DTC,30000,Sandstone,65000,Shale,1.0,31677,17,0.138307,0.333549,0.195243,0.097406,0.291693,0.000839,0.102529,0.941176,0.171985,0.083523,0.257849,0.001068,0.024262,0.011518,0.028231,0.059068,0.041387,active_representation_comparison,True,5000,0.011688,0.011251,True,amplified
1095,fold_4__feature_GR__source_65000_Shale__target_70000_Limestone,4,GR,65000,Shale,70000,Limestone,1.0,142571,17,0.872009,1.051579,0.179570,0.063477,0.306624,0.010986,0.202877,0.705882,0.035345,-0.073292,0.147018,0.611221,0.106116,0.070577,-0.002101,0.070380,0.015217,active_representation_comparison,True,5000,0.057861,0.808142,False,amplified
935,fold_2__feature_RDEP__source_70000_Limestone__target_65000_Shale,2,RDEP,70000,Limestone,65000,Shale,1.0,9844,20,-0.165184,0.013743,0.178927,0.082639,0.292995,0.001432,0.156225,0.800000,0.180623,0.086398,0.288177,0.001209,0.008391,-0.057358,-0.002213,0.020060,0.010760,active_representation_comparison,True,5000,0.014601,0.011941,True,amplified
623,fold_5__fea

Top suppressed active conditions with at least 10 wells:


,condition_id,outer_fold,feature,source_code,source_name,target_code,target_name,alpha,rows,wells,raw8_well_balanced_calibrated_delta_log_loss,legacy_well_balanced_calibrated_delta_log_loss,well_balanced_calibrated_representation_amplification,calibrated_CI_lower,calibrated_CI_upper,calibrated_wilcoxon_pvalue,median_per_well_calibrated_amplification,fraction_wells_positive_calibrated_amplification,well_balanced_raw_representation_amplification,raw_CI_lower,raw_CI_upper,raw_wilcoxon_pvalue,well_balanced_accuracy_harm_amplification,well_balanced_any_flip_amplification,well_balanced_targeted_flip_amplification,well_balanced_source_probability_harm_amplification,well_balanced_target_probability_change_difference,comparison_scope,stable_condition_support,bootstrap_replicates,calibrated_qvalue,raw_qvalue,calibrated_FDR_significant,calibrated_amplification_class
851,fold_2__feature_GR__source_70000_Limestone__target_65000_Shale,2,GR,70000,Limestone,65000,Shale,1.0,9844,20,0.887865,0.603747,-0.284117,-0.505372,-0.052174,0.012079,-0.301109,0.300000,-0.362272,-0.569133,-0.139957,0.002325,-0.032566,-0.114623,0.049216,0.029304,-0.001037,active_representation_comparison,True,5000,0.060588,0.017920,False,suppressed
863,fold_2__feature_NPHI__source_30000_Sandstone__target_65000_Shale,2,NPHI,30000,Sandstone,65000,Shale,1.0,26875,20,0.684354,0.452438,-0.231916,-0.370627,-0.115477,0.000851,-0.192640,0.250000,-0.263666,-0.391853,-0.151109,0.000168,-0.127631,-0.123815,-0.112498,-0.071725,-0.032105,active_representation_comparison,True,5000,0.011688,0.004736,True,suppressed
607,fold_5__feature_NPHI__source_65000_Shale__target_30000_Sandstone,5,NPHI,65000,Shale,30000,Sandstone,1.0,84211,20,0.396652,0.207214,-0.189438,-0.285641,-0.096756,0.000168,-0.095514,0.050000,-0.187113,-0.287283,-0.098252,0.000210,-0.103664,-0.151607,-0.125975,-0.090099,-0.078768,active_representation_comparison,True,5000,0.005260,0.004736,True,suppressed
99,fold_1__feature_NPHI__source_65000_Shale__target_30000_Sandstone,1,NPHI,65000,Shale,30000,Sandstone,1.0,96664,20,0.361810,0.173173,-0.188637,-0.237281,-0.140013,0.000010,-0.205438,0.050000,-0.168728,-0.212257,-0.125446,0.000006,-0.052415,-0.057415,-0.031938,-0.113733,-0.056866,active_representation_comparison,True,5000,0.001507,0.000904,True,suppressed
327,fold_3__feature_GR__source_70000_Limestone__target_65000_Shale,3,GR,70000,Limestone,65000,Shale,1.0,11090,20,0.893530,0.704954,-0.188576,-0.480264,0.062454,0.294252,-0.142118,0.400000,-0.353786,-0.637633,-0.120060,0.008308,0.040192,0.049133,0.039398,0.018562,0.023086,active_representation_comparison,True,5000,0.537478,0.041023,False,inconclusive
107,fold_1__feature_NPHI__source_65000_Shale__target_70000_Limestone,1,NPHI,65000,Shale,70000,Limestone,1.0,96664,20,0.457983,0.275761,-0.182222,-0.238502,-0.117222,0.000105,-0.198470,0.050000,-0.155977,-0.207400,-0.099593,0.000210,-0.064707,-0.058987,-0.018476,-0.102564,-0.048705,active_representation_comparison,True,5000,0.005260,0.004736,True,suppressed
615,fold_5__feature_NPHI__source_65000_Shale__target_70000_Limestone,5,NPHI,65000,Shale,70000,Limestone,1.0,84211,20,0.513790,0.335242,-0.178548,-0.286279,-0.074432,0.002712,-0.115353,0.250000,-0.180569,-0.293418,-0.074769,0.001690,-0.127009,-0.166429,-0.013188,-0.078070,-0.039382,active_representation_comparison,True,5000,0.020406,0.014053,True,suppressed
603,fold_5__feature_NPHI__source_30000_Sandstone__target_65000_Shale,5,NPHI,30000,Sandstone,65000,Shale,1.0,22153,20,0.530661,0.363059,-0.167602,-0.250069,-0.082700,0.001986,-0.157770,0.150000,-0.211893,-0.303843,-0.127822,0.000082,-0.097117,-0.126109,-0.064568,-0.074432,-0.036828,active_representation_comparison,True,5000,0.016958,0.003702,True,suppressed
95,fold_1__feature_NPHI__source_30000_Sandstone__target_65000_Shale,1,NPHI,30000,Sandstone,65000,Shale,1.0,23810,20,0.563812,0.416640,-0.147173,-0.246567,-0.063581,0.001986,-0.084284,0.250000,-0.212211,-0.310902,-0.127942,0.000048,-0.122387,-0.124687,-0.106514,-0.065278,-0.

# 11. Stage gate

In [13]:
stage_gate = {
    "official_test_unused": True,
    "06A_full_passed": bool(
        upstream_gate_audit[
            "06A_full_passed"
        ]
    ),
    "06B_full_passed": bool(
        upstream_gate_audit[
            "06B_full_passed"
        ]
    ),
    "summary_pairing_complete": bool(
        pairing_audit[
            "summary_pairing_complete"
        ]
    ),
    "per_well_pairing_complete": bool(
        pairing_audit[
            "per_well_pairing_complete"
        ]
    ),
    "summary_row_counts_match": bool(
        pairing_audit[
            "summary_row_counts_match"
        ]
    ),
    "summary_well_counts_match": bool(
        pairing_audit[
            "summary_well_counts_match"
        ]
    ),
    "per_well_eligible_rows_match": bool(
        pairing_audit[
            "per_well_eligible_rows_match"
        ]
    ),
    "expected_condition_dose_count": 1276,
    "observed_condition_dose_count": int(
        len(
            condition_amplification
        )
    ),
    "condition_dose_count_correct": (
        len(
            condition_amplification
        )
        == 1276
    ),
    "expected_paired_per_well_rows": 20928,
    "observed_paired_per_well_rows": int(
        len(
            paired_per_well
        )
    ),
    "paired_per_well_row_count_correct": (
        len(
            paired_per_well
        )
        == 20928
    ),
    "all_paired_effects_finite": bool(
        np.isfinite(
            paired_per_well[
                finite_columns
            ].to_numpy(float)
        ).all()
    ),
    "PEF_SP_legacy_effects_exact_zero": bool(
        structural_exact_zero
    ),
    "active_and_structural_scopes_separated": bool(
        set(
            condition_amplification[
                "comparison_scope"
            ].unique()
        )
        == {
            "active_representation_comparison",
            "structural_feature_exclusion",
        }
    ),
    "well_balanced_primary_estimand": True,
    "positive_amplification_means_legacy_more_harmed": True,
    "no_model_refit": True,
    "no_calibration_refit": True,
    "run_mode": "full",
}

stage_gate[
    "overall_representation_amplification_stage_passed"
] = bool(
    all(
        value
        for key, value in stage_gate.items()
        if (
            isinstance(
                value,
                bool,
            )
            and key
            != "overall_representation_amplification_stage_passed"
        )
    )
)

print(
    json.dumps(
        stage_gate,
        indent=2,
    )
)

if not stage_gate[
    "overall_representation_amplification_stage_passed"
]:
    raise RuntimeError(
        "06C stage gate failed."
    )

{
  "official_test_unused": true,
  "06A_full_passed": true,
  "06B_full_passed": true,
  "summary_pairing_complete": true,
  "per_well_pairing_complete": true,
  "summary_row_counts_match": true,
  "summary_well_counts_match": true,
  "per_well_eligible_rows_match": true,
  "expected_condition_dose_count": 1276,
  "observed_condition_dose_count": 1276,
  "condition_dose_count_correct": true,
  "expected_paired_per_well_rows": 20928,
  "observed_paired_per_well_rows": 20928,
  "paired_per_well_row_count_correct": true,
  "all_paired_effects_finite": true,
  "PEF_SP_legacy_effects_exact_zero": true,
  "active_and_structural_scopes_separated": true,
  "well_balanced_primary_estimand": true,
  "positive_amplification_means_legacy_more_harmed": true,
  "no_model_refit": true,
  "no_calibration_refit": true,
  "run_mode": "full",
  "overall_representation_amplification_stage_passed": true
}


In [14]:
upstream_audit_df = pd.DataFrame([
    {
        "audit_item": key,
        "passed": bool(
            value
        ),
    }
    for key, value in upstream_gate_audit.items()
])

pairing_audit_df = pd.DataFrame([
    {
        "audit_item": key,
        "value": value,
    }
    for key, value in pairing_audit.items()
])

upstream_audit_df.to_csv(
    OUTPUT_DIR
    / "00_upstream_gate_audit.csv",
    index=False,
)

pairing_audit_df.to_csv(
    OUTPUT_DIR
    / "01_pairing_audit.csv",
    index=False,
)

paired_summary.to_csv(
    OUTPUT_DIR
    / "02_paired_condition_dose_inputs.csv",
    index=False,
)

paired_per_well.to_parquet(
    OUTPUT_DIR
    / "03_paired_per_well_effects.parquet",
    index=False,
)

paired_per_well.to_csv(
    OUTPUT_DIR
    / "03_paired_per_well_effects.csv",
    index=False,
)

condition_amplification.to_csv(
    OUTPUT_DIR
    / "04_condition_dose_representation_amplification.csv",
    index=False,
)

per_well_feature_alpha.to_csv(
    OUTPUT_DIR
    / "05_per_well_feature_alpha_amplification.csv",
    index=False,
)

feature_amplification.to_csv(
    OUTPUT_DIR
    / "06_feature_alpha_representation_amplification.csv",
    index=False,
)

fold_feature_alpha.to_csv(
    OUTPUT_DIR
    / "07_fold_feature_alpha_amplification.csv",
    index=False,
)

dose_response_summary.to_csv(
    OUTPUT_DIR
    / "08_feature_amplification_dose_response.csv",
    index=False,
)

structural_exclusion_summary.to_csv(
    OUTPUT_DIR
    / "09_structural_exclusion_controls.csv",
    index=False,
)

top_amplified_alpha1_all.to_csv(
    OUTPUT_DIR
    / "10_top_amplified_conditions_alpha1_all.csv",
    index=False,
)

top_suppressed_alpha1_all.to_csv(
    OUTPUT_DIR
    / "11_top_suppressed_conditions_alpha1_all.csv",
    index=False,
)

top_amplified_alpha1_stable.to_csv(
    OUTPUT_DIR
    / "12_top_amplified_conditions_alpha1_stable_support.csv",
    index=False,
)

top_suppressed_alpha1_stable.to_csv(
    OUTPUT_DIR
    / "13_top_suppressed_conditions_alpha1_stable_support.csv",
    index=False,
)

with open(
    OUTPUT_DIR
    / "14_stage_gate.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        json_safe(
            stage_gate
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )

run_configuration = {
    "stage": (
        "06C_raw_vs_legacy_representation_amplification"
    ),
    "run_mode": "full",
    "official_test_used": False,
    "raw8_source_stage": (
        "06A_raw_RF_class_conditional_transport_interventions/full"
    ),
    "legacy_source_stage": (
        "06B_legacy_RF_class_conditional_transport_interventions/full"
    ),
    "primary_effect": (
        "legacy calibrated delta log loss minus "
        "Raw-8 calibrated delta log loss"
    ),
    "primary_estimand": (
        "paired within WELL, then equal mean across WELLs"
    ),
    "condition_inference": (
        "paired WELL bootstrap confidence interval and "
        "paired Wilcoxon signed-rank test"
    ),
    "feature_inference": (
        "average available directed conditions within each WELL, "
        "then bootstrap and Wilcoxon across unique WELLs"
    ),
    "condition_FDR": (
        "Benjamini-Hochberg separately within each alpha"
    ),
    "feature_FDR": (
        "Benjamini-Hochberg separately within each alpha; "
        "active-feature and all-feature q-values both saved"
    ),
    "active_features": ACTIVE_FEATURES,
    "structural_exclusion_features": (
        STRUCTURAL_EXCLUSION_FEATURES
    ),
    "structural_exclusion_interpretation": (
        "PEF and SP are present in Raw-8 but absent from the "
        "final Legacy representation; their comparison is not "
        "interpreted as descendant amplification"
    ),
    "minimum_wells_for_stable_condition_ranking": int(
        MINIMUM_WELLS_FOR_STABLE_CONDITION_RANKING
    ),
    "bootstrap_replicates": int(
        BOOTSTRAP_REPLICATES
    ),
    "model_refitted": False,
    "calibration_refitted": False,
    "next_stage": (
        "robust_model_candidate_design_and_nested_grouped_training"
    ),
}

with open(
    OUTPUT_DIR
    / "15_run_configuration.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        json_safe(
            run_configuration
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )

input_hashes = {
    "raw8_summary": sha256_file(
        RAW8_SUMMARY_PATH
    ),
    "raw8_per_well": sha256_file(
        RAW8_PER_WELL_PATH
    ),
    "raw8_gate": sha256_file(
        RAW8_GATE_PATH
    ),
    "raw8_config": sha256_file(
        RAW8_CONFIG_PATH
    ),
    "legacy_summary": sha256_file(
        LEGACY_SUMMARY_PATH
    ),
    "legacy_per_well": sha256_file(
        LEGACY_PER_WELL_PATH
    ),
    "legacy_gate": sha256_file(
        LEGACY_GATE_PATH
    ),
    "legacy_config": sha256_file(
        LEGACY_CONFIG_PATH
    ),
}

with open(
    OUTPUT_DIR
    / "16_input_hashes.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        input_hashes,
        file,
        ensure_ascii=False,
        indent=2,
    )

environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
}

with open(
    OUTPUT_DIR
    / "17_environment.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    "Outputs saved in:",
    OUTPUT_DIR,
)

Outputs saved in: <ARTIFACT_ROOT>/06C_raw_vs_legacy_representation_amplification/full
